# USL Championship — Team Data

League-wide metrics using the ASA public API.

In [ ]:
from typing import Final

import pandas as pd
from itscalledsoccer.client import AmericanSoccerAnalysis

pd.options.display.max_columns = 999
pd.set_option("expand_frame_repr", False)

asa_client = AmericanSoccerAnalysis()

LEAGUE: Final[str] = "uslc"
FOCUS_TEAM: Final[str] = "LOU"  # change to any team abbreviation to repoint the notebook

## Data Fetch

In [ ]:
team_xg = asa_client.get_team_xgoals(
    leagues=[LEAGUE],
    split_by_seasons=True,
    stage_name="Regular Season",
)

team_xp = asa_client.get_team_xpass(
    leagues=[LEAGUE],
    split_by_seasons=True,
    stage_name="Regular Season",
)

team_ga = asa_client.get_team_goals_added(
    leagues=[LEAGUE],
    split_by_seasons=True,
    stage_name="Regular Season",
)

## Data Cleaning

Resolve `team_id` to abbreviation, then join xGoals, xPass, and Goals Added
into a single frame.


In [ ]:
teams = asa_client.get_teams(leagues=[LEAGUE])
team_map: dict[str, str] = dict(zip(teams["team_id"], teams["team_abbreviation"]))

for _df in [team_xg, team_xp, team_ga]:
    _df["team_name"] = _df["team_id"].map(team_map)
    _df.drop(columns=["team_id"], inplace=True)

# team_ga arrives with a nested `data` column — a list of per-action-type dicts
# (Dribbling, Fouling, Interrupting, Passing, Receiving, Shooting, Claiming).
# Explode + json_normalize unpacks those into one row per (team, season, action),
# then pivot reshapes into ga_for_<action> / ga_against_<action> columns.
_ga_long = team_ga.explode("data", ignore_index=True)
_ga_long = _ga_long.join(pd.json_normalize(_ga_long.pop("data")))
team_ga_wide = _ga_long.pivot(
    index=["team_name", "season_name", "minutes"],
    columns="action_type",
    values=["goals_added_for", "goals_added_against"],
)
team_ga_wide.columns = [
    f"ga_{direction.rsplit('_', 1)[-1]}_{action.lower()}"
    for direction, action in team_ga_wide.columns
]
team_ga_wide = team_ga_wide.reset_index()

# Totals across all action types, plus net differential.
_ga_for_cols = [c for c in team_ga_wide.columns if c.startswith("ga_for_")]
_ga_against_cols = [c for c in team_ga_wide.columns if c.startswith("ga_against_")]
team_ga_wide["ga_for_total"] = team_ga_wide[_ga_for_cols].sum(axis=1)
team_ga_wide["ga_against_total"] = team_ga_wide[_ga_against_cols].sum(axis=1)
team_ga_wide["ga_difference"] = (
    team_ga_wide["ga_for_total"] - team_ga_wide["ga_against_total"]
)
_ga_numeric_cols = [c for c in team_ga_wide.columns if c.startswith("ga_")]
team_ga_wide[_ga_numeric_cols] = team_ga_wide[_ga_numeric_cols].round(2)

# Merge xPass then the expanded Goals Added frame into xGoals, each time keeping
# only source-exclusive columns to prevent suffix conflicts on shared columns.
_join_on = ["team_name", "season_name"]
for _source in [team_xp, team_ga_wide]:
    _cols = _join_on + [c for c in _source.columns if c not in team_xg.columns]
    team_xg = team_xg.merge(_source[_cols], on=_join_on, how="left")

team_stats = (
    team_xg.copy()
    .sort_values(["team_name", "season_name"])
    .reset_index(drop=True)
)

# team_stats.info()
display(team_stats.head())

## Per-Game Metrics

Normalize cumulative totals by `count_games` so teams with different schedules
are directly comparable.

In [ ]:
# Shots — per game
team_stats["shots_per_game"] = (
    team_stats["shots_for"] / team_stats["count_games"]
).round(2)
team_stats["shots_against_per_game"] = (
    team_stats["shots_against"] / team_stats["count_games"]
).round(2)

# Goals — per game
team_stats["goals_per_game"] = (
    team_stats["goals_for"] / team_stats["count_games"]
).round(2)
team_stats["goals_against_per_game"] = (
    team_stats["goals_against"] / team_stats["count_games"]
).round(2)

# xG — per game
team_stats["xg_per_game"] = (
    team_stats["xgoals_for"] / team_stats["count_games"]
).round(2)
team_stats["xga_per_game"] = (
    team_stats["xgoals_against"] / team_stats["count_games"]
).round(2)

# Overperformance vs expected (positive = better than model predicts)
team_stats["finishing_overperformance"] = (
    team_stats["goals_for"] - team_stats["xgoals_for"]
).round(2)
team_stats["defensive_overperformance"] = (
    team_stats["xgoals_against"] - team_stats["goals_against"]
).round(2)

# Points — per game, expected, overperformance
team_stats["pts_per_game"] = (
    team_stats["points"] / team_stats["count_games"]
).round(2)
team_stats["xpts_per_game"] = (
    team_stats["xpoints"] / team_stats["count_games"]
).round(2)
team_stats["points_overperformance"] = (
    team_stats["points"] - team_stats["xpoints"]
).round(2)

# Passes — per game
team_stats["passes_per_game"] = (
    team_stats["attempted_passes_for"] / team_stats["count_games"]
).round(1)

## Advanced Metrics

Derived analytical layers built on top of the per-game cell: shot quality,
possession share, g+ per 96 minutes, season-over-season deltas, and
within-season percentile rank.

In [ ]:
# Shot quality — xG generated per shot taken/allowed. Separates teams that
# create good looks from teams that just shoot often.
team_stats["xg_per_shot_for"] = (
    team_stats["xgoals_for"] / team_stats["shots_for"]
).round(3)
team_stats["xg_per_shot_against"] = (
    team_stats["xgoals_against"] / team_stats["shots_against"]
).round(3)

# Possession share proxy — pass-volume share vs opponent. ASA does not
# publish possession %, but pass share is a strong proxy and pairs naturally
# with the existing vertical-distance metrics.
team_stats["pass_share"] = (
    team_stats["attempted_passes_for"]
    / (team_stats["attempted_passes_for"] + team_stats["attempted_passes_against"])
).round(3)

In [ ]:
# g+ per 96 minutes — ASA's standard rate convention. Normalizes across
# seasons with different game counts (15-game 2020 vs 34-game 2024).
_ga_cols = [c for c in team_stats.columns if c.startswith("ga_")]
_per96 = 96 / team_stats["minutes"]
for _c in _ga_cols:
    team_stats[f"{_c}_p96"] = (team_stats[_c] * _per96).round(3)

In [ ]:
# Season-over-season change per team. First season for each team yields NaN.
# Relies on team_stats being sorted by (team_name, season_name) — enforced in
# the Data Cleaning cell.
_delta_cols = [
    "pts_per_game", "xpts_per_game",
    "goals_per_game", "xg_per_game",
    "goals_against_per_game", "xga_per_game",
    "ga_difference",
]
for _c in _delta_cols:
    team_stats[f"{_c}_yoy"] = (
        team_stats.groupby("team_name")[_c].diff().round(2)
    )

In [ ]:
# League percentile rank within season (0–1, higher = better). For defensive
# metrics (lower raw value is better), rank descending so the stingiest defense
# still scores near 1.0.
_rank_high_is_good = [
    "pts_per_game", "xpts_per_game",
    "goals_per_game", "xg_per_game",
    "xg_per_shot_for",
    "pass_share",
    "ga_for_total", "ga_difference",
]
_rank_low_is_good = [
    "goals_against_per_game", "xga_per_game",
    "xg_per_shot_against",
    "ga_against_total",
]
for _c in _rank_high_is_good:
    team_stats[f"{_c}_pct"] = (
        team_stats.groupby("season_name")[_c].rank(pct=True).round(3)
    )
for _c in _rank_low_is_good:
    team_stats[f"{_c}_pct"] = (
        team_stats.groupby("season_name")[_c]
        .rank(pct=True, ascending=False)
        .round(3)
    )

---

In [ ]:
_col_order = [
    # Identity
    "team_name", "season_name", "count_games", "minutes",
    # Points
    "points", "xpoints", "pts_per_game", "xpts_per_game", "points_overperformance",
    "pts_per_game_pct", "xpts_per_game_pct",
    "pts_per_game_yoy", "xpts_per_game_yoy",
    # Scoring
    "goals_for", "goals_per_game", "xgoals_for", "xg_per_game",
    "finishing_overperformance", "shots_for", "shots_per_game",
    "xg_per_shot_for", "xg_per_shot_for_pct",
    "goals_per_game_pct", "xg_per_game_pct",
    "goals_per_game_yoy", "xg_per_game_yoy",
    # Conceded
    "goals_against", "goals_against_per_game", "xgoals_against", "xga_per_game",
    "defensive_overperformance", "shots_against", "shots_against_per_game",
    "xg_per_shot_against", "xg_per_shot_against_pct",
    "goals_against_per_game_pct", "xga_per_game_pct",
    "goals_against_per_game_yoy", "xga_per_game_yoy",
    # Differential
    "goal_difference", "xgoal_difference", "goal_difference_minus_xgoal_difference",
    # Passing — offensive
    "attempted_passes_for", "passes_per_game", "pass_share", "pass_share_pct",
    "pass_completion_percentage_for", "xpass_completion_percentage_for",
    "passes_completed_over_expected_for", "passes_completed_over_expected_p100_for",
    "avg_vertical_distance_for",
    # Passing — against
    "attempted_passes_against",
    "pass_completion_percentage_against", "xpass_completion_percentage_against",
    "passes_completed_over_expected_against", "passes_completed_over_expected_p100_against",
    "avg_vertical_distance_against",
    # Passing — difference
    "passes_completed_over_expected_difference", "avg_vertical_distance_difference",
    # Goals Added — totals and net
    "ga_for_total", "ga_against_total", "ga_difference",
    "ga_for_total_pct", "ga_against_total_pct", "ga_difference_pct",
    "ga_difference_yoy",
    # Goals Added — totals per 96
    "ga_for_total_p96", "ga_against_total_p96", "ga_difference_p96",
    # Goals Added — offensive by action type
    "ga_for_passing", "ga_for_receiving", "ga_for_dribbling", "ga_for_shooting",
    "ga_for_interrupting", "ga_for_fouling", "ga_for_claiming",
    # Goals Added — offensive by action type per 96
    "ga_for_passing_p96", "ga_for_receiving_p96", "ga_for_dribbling_p96",
    "ga_for_shooting_p96", "ga_for_interrupting_p96", "ga_for_fouling_p96",
    "ga_for_claiming_p96",
    # Goals Added — defensive by action type
    "ga_against_passing", "ga_against_receiving", "ga_against_dribbling",
    "ga_against_shooting", "ga_against_interrupting", "ga_against_fouling",
    "ga_against_claiming",
    # Goals Added — defensive by action type per 96
    "ga_against_passing_p96", "ga_against_receiving_p96", "ga_against_dribbling_p96",
    "ga_against_shooting_p96", "ga_against_interrupting_p96",
    "ga_against_fouling_p96", "ga_against_claiming_p96",
]
team_stats = team_stats[_col_order]
display(team_stats.head())

In [ ]:
team_stats.to_parquet("../data/team_stats.parquet", index=False)

-----------------

## Focus Team

Isolate all seasons for `FOCUS_TEAM`. Change `FOCUS_TEAM` in Cell 1 to
repoint this section at any team in the dataset.


In [ ]:
lou = team_stats[team_stats["team_name"] == FOCUS_TEAM].copy()
display(lou)

## Usage Example

In [ ]:
cols = [
    "team_name",
    "season_name",
    "count_games",
    "pts_per_game",
    "goals_per_game",
    "goals_against_per_game",
]
example = team_stats[cols].sort_values("pts_per_game", ascending=False)
example = example[example["count_games"] >= example["count_games"].max() / 2]
display(example.head(10))